# Implementation of a transformer model
In this exercise we will implement the transformer model (an architecture based on LLama2). The squeleton of the code is provided and you should complete it (mostly self attention mechanism).

In [234]:
import time 
from time import gmtime, strftime
import math
import shutil
import os



# neural network utilities
import torch
from torch import nn
from torch import optim
from torch.nn import functional as F
from torch.utils.data import DataLoader

# dataset an tokenization
from datasets import load_dataset
from transformers import AutoTokenizer

from rope_embedding import RoPE


## Self Attention with no mask

Let consider $Q, K, V \in R^{B\times H \times L \times N}$ for each batch element (B) for each heads (H) we want to compute $$A = \frac{Q_{bh}.K_{bh}^{t}}{\sqrt{N}}$$

Attention weights are defined by $$ 
softmax(A_i) = \begin{pmatrix}
\frac{e^{A_{i1}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} & 
\frac{e^{A_{i2}}}{\sum\limits_{j=1}^L  e^{A_{ij}}} & 
\dots &
\frac{e^{A_{iL}}}{\sum\limits_{j=1}^L  e^{A_{ij}}}
\end{pmatrix}$$ 
And $$ 
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} & 
\frac{e^{A_{12}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} & 
\dots &
\frac{e^{A_{1L}}}{\sum\limits_{j=1}^L  e^{A_{1j}}} \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} & 
\frac{e^{A_{22}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} & 
\dots &
\frac{e^{A_{2l}}}{\sum\limits_{j=1}^L  e^{A_{2j}}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\dots &
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}}
\end{pmatrix}$$ 


Notice that the function softmax can be used to compute this matrix [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html#torch.softmax).


## Masked Attention
In many cases, certains tokens will not compute attention with later tokens (padding or decoding approaches). Typically in decoder only architectures we would have the attention weights matrix only considering lower diagonal values :

$$ 
Softmax(A) = \begin{pmatrix}
\frac{e^{A_{11}}}{\sum\limits_{j=1}^1  e^{A_{1j}}} & 
0 & 
\dots & 0 &
0 \\
\frac{e^{A_{21}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} & 
\frac{e^{A_{22}}}{\sum\limits_{j=1}^2  e^{A_{2j}}} & 
\dots & 0 &
0 \\
\vdots & \vdots & \ddots & \vdots \\
\frac{e^{A_{{L-1}1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} & 
\frac{e^{A_{{L-1}2}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}} & 
\dots &
\frac{e^{A_{L-1L-1}}}{\sum\limits_{j=1}^{L-1}  e^{A_{L-1j}}}& 0 \\
\frac{e^{A_{L1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{L2}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\dots &
\frac{e^{A_{LL-1}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} & 
\frac{e^{A_{LL}}}{\sum\limits_{j=1}^L  e^{A_{Lj}}} \\
\end{pmatrix}$$ 

A simple implementation would consist in setting the upper diagonal of A to $-\infty$ and apply the softmax. We will thus consider a matrix name $M$ (or `mask` in the code) having upper diagonal element to $-\infty$ : 
$$ M = 
\begin{pmatrix}
0& 
-\infty & 
\dots & -\infty&
-\infty \\
0 & 
0 & 
\dots & -\infty &
-\infty \\
\vdots & \vdots & \ddots & \vdots \\
0 & 
0 & 
\dots &
0& -\infty \\
0 & 
0 & 
\dots &
0 & 
0\\
\end{pmatrix}
$$

And apply softmax on $A + M$

In [235]:
def multi_head_attention(
    q : torch.FloatTensor,
    k : torch.FloatTensor,
    v : torch.FloatTensor,
    mask : torch.FloatTensor
):
    """
    Given Q,K,V and the attention mask compute cross attention

    The shape of the matrix are the following:
    B: The batch size (number of examples)
    Q: The size of the decoder sequence
    K: The size of the encoder sequence
    H: Then number of attention heads
    N: The embedding size 
    
    Parameters:
        q : torch.FloatTensor
            The query matrix of shape BxHxQxN
        k : torch.FloatTensor
            The key matrix of shape BxHxKxN
        v : torch.FloatTensor
            The value matrix of shape BxHxKxN
        mask : torch.FloatTensor
            The attention mask of shape BxQxK,
            the masked elements are set to -inf 
            else elements are set to 0
    Return : (torch.Tensor, torch.Tensor)
        return two tensor the first containing 
        the attention weights (QK^t) and the second 
        the result of the attention
    """
    attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.size(-1))
    if mask is not None:
        mask = mask.to(device=attention_scores.device, dtype=attention_scores.dtype)
        attention_scores = attention_scores + mask

    attention_weights = torch.softmax(attention_scores, dim=-1)
    attention_output = torch.matmul(attention_weights, v)
    return attention_weights, attention_output


In [236]:
q, k, v = (torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4), torch.ones(1, 1, 3, 4))
mask = torch.triu(-q.new_ones(1, 3, 3) *  torch.inf, diagonal =  1)
attention_weigths, attention_output = multi_head_attention(q, k, v, mask)

assert(torch.all(attention_weigths == torch.Tensor([
          [[[1., 0., 0.],
            [1/2, 1/2, 0.],
            [1/3, 1/3, 1/3]]]])))

##  Create the attention module

In pytorch module are block of neural networks that inherit from the class `nn.Module`.  In the following you willbe asked to complete the attention code. The attention module compute the Query $Q$, the Keys $K$ and Values $V$ for each heads and compute attention. The forward methods is the method implementing these operations. In this lab we use a mult-head attention, meaning that we compute attention for each heads, that can be done with a for loop or using batched operation (it is up to you). Also as we follow the Llama 2 architecture, we use for positional embeding the RoPe positonal encoding that is applied on the two matrix $Q$ and $K$. The forward function should compute for an input $X \in \mathbb{R}^{L \times N}$ (in the decoder only case) the following:

1. $Q_h = RoPE(W^q_{h}X^{\intercal})$ (for all heads h) with $W^q_h \in \mathbb{R}^{N \times N}$
2. $K_h = RoPE(W^k_{h}X^{\intercal})$ (for all heads h)
3. $V_h = W^v_{h}X^{\intercal}$ (for all heads h)
4. $A_h =  Attention(Q_h, K_h, V_h)$ (for all heads h)
5. $A$ = [A_1, A_2, \dots A_H] (the concatenation along the last dimension)
6. $O = W^oA^{\intercal}$ with $w^{o} \in \mathbb{R}^{N \times NH}$ (NH being the scalar multplication of N and H)

Notice that Q, K and V can be computed without a loop using $W_{q}$ (respectively for $W_k$ and $W_v$) .


In [264]:
def causal_mask(seq_len, past_len=0, device=None):
    query_positions = past_len + torch.arange(seq_len, device=device)
    key_positions = torch.arange(past_len + seq_len, device=device)

    mask = torch.zeros(
        seq_len,
        past_len + seq_len,
        device=device,
    )

    mask = mask.masked_fill(
        key_positions.unsqueeze(0) > query_positions.unsqueeze(1),
        float("-inf"),
    )
    torch.triu(
        torch.full((seq_len, past_len + seq_len), float("-inf"), device=device),
        diagonal=1,
    )
    return mask

In [265]:

class RoPEAttentionModule(nn.Module):
    """ A cross/self attention pytorch module.
    
    """
    def __init__(self, input_dim, output_dim, num_heads=1):
        super().__init__()
        
        self.Wq, self.Wk, self.Wv  = (nn.Linear(input_dim, output_dim * num_heads) for _ in range(3))

        self.output_dim = output_dim
        self.num_heads = num_heads
        self.Wo = nn.Linear(output_dim * num_heads, input_dim)
        self.rope_func = RoPE(output_dim)
    
    
    def forward(
        self,
        x: torch.Tensor,
        attention_mask: torch.BoolTensor = None,
        y: torch.Tensor = None,
        decoder_mask=True,
        k_cache=None,
        v_cache=None,
        start_cache=0,
    ):
        # dimensions
        B = x.size(0)
        L = x.size(1)
        H = self.num_heads
        dh = self.output_dim

        # Decoder-only : self-attention with cache
        if decoder_mask:
            # Computing q, k, v
            q = self.Wq(x).reshape(B, L, H, dh).transpose(1, 2)
            k_new = self.Wk(x).reshape(B, L, H, dh).transpose(1, 2)
            v_new = self.Wv(x).reshape(B, L, H, dh).transpose(1, 2)

            # RoPE with qbsolute position of new tokens
            q = self.rope_func(q, start_pos_id=start_cache)
            k_new = self.rope_func(k_new, start_pos_id=start_cache)

            # Concatenate with the past tokens
            past_len = 0 if k_cache is None else k_cache.size(2)

            if k_cache is not None:
                k = torch.cat([k_cache, k_new], dim=2)
                v = torch.cat([v_cache, v_new], dim=2)
            else:
                k = k_new
                v = v_new

            # causal mask compatible with the cache
            attention_mask = causal_mask(
                L,
                past_len=past_len,
                device=x.device,
            )

            # Final projection
            attention_weights, attention_output = multi_head_attention(
                q, k, v, attention_mask
            )

            attention_output = attention_output.transpose(1, 2).reshape(
                B, L, H * dh
            )
            output = self.Wo(attention_output)

            return attention_weights, output, k, v

        # Cross-attention
        k = self.Wk(x).reshape(B, L, H, dh).transpose(1, 2)
        v = self.Wv(x).reshape(B, L, H, dh).transpose(1, 2)

        Ly = y.size(1)
        q = self.Wq(y).reshape(B, Ly, H, dh).transpose(1, 2)

        q = self.rope_func(q)
        k = self.rope_func(k)

        attention_weights, attention_output = multi_head_attention(
            q, k, v, attention_mask
        )

        attention_output = attention_output.transpose(1, 2).reshape(
            B, Ly, H * dh
        )
        output = self.Wo(attention_output)

        return attention_weights, output


## Trasnsformer FeedForward

The FeedForward Network is defined using Llama 2 architecture as following for $X \in \mathbb{R^{L\times N}}$ :


1. $G = SiLU(W^gX^\intercal)$ with $W^g \in \mathbb{R}^{M \times N}$ (see SiLU in pytorch [documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.SiLU.html))
2. $U = W^u X^\intercal$ with $W^u  \in \mathbb{R}^{M \times N}$
3. $I = G \odot I$ (Hadamard multiplication)
4. $O = W^{o} I^{\intercal}$ with $W_{o} \in \mathbb{R}^{N \times M}$






In [266]:
class TransformerFeedForward(nn.Module):
    def __init__(self, embed_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(embed_size, intermediate_size)
        self.up_proj = nn.Linear(intermediate_size, intermediate_size) # Correction of embed_size -> intermediate_size
        self.down_proj = nn.Linear(intermediate_size, embed_size)
        self.gate_func = nn.SiLU()
        
    def forward(self, x):
        x = self.gate_proj(x)
        x = self.up_proj(x)
        x = self.down_proj(x)
        return self.gate_func(x)

## Transformer block
We can now create the block of the 

Let consider $x_0 \in \mathbb{R}^{B \times L \times N}$, the decoder block is given by : 

* $x_1 = LN_1(x_0)$ (apply layer norm)
* $x_2 = Attention(x_1)$ (apply attention)
* $x_3 = x_0 + x_2$ (adding residual)
* $x_4 = LN_2(x_3)$
* $y = FF(x_4) + x_4$


In [267]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_size, intermediate_size, num_heads):
        super().__init__()
        self.attention_module = RoPEAttentionModule(embed_size, embed_size, num_heads=num_heads)
        self.feed_forward = TransformerFeedForward(embed_size, intermediate_size)
        
        self.attention_layer_norm = nn.RMSNorm(embed_size)
        self.feed_forward_layer_norm = nn.RMSNorm(embed_size)
        
    
    def forward(
        self,
        x,
        attention_mask=None,
        k_cache=None,
        v_cache=None,
        start_cache=0,
        layer=0,
    ):
        x1 = self.attention_layer_norm(x)

        # Recovering the caches of the block
        layer_k = None if k_cache is None else k_cache[layer]
        layer_v = None if v_cache is None else v_cache[layer]

        # Attention with KV cache
        _, x2, new_k, new_v = self.attention_module(
            x1,
            attention_mask=attention_mask,
            decoder_mask=True,
            k_cache=layer_k,
            v_cache=layer_v,
            start_cache=start_cache,
        )

        # Residual connection
        x3 = x + x2

        # Feed-forward and Second residual connection
        x4 = self.feed_forward_layer_norm(x3)
        output = self.feed_forward(x4) + x3

        return output, new_k, new_v


## The Model

The model contain a stack of block on input embedding, it applies the different transformation and should return both the output embeddings and the logits (size of the vocabulary).

It should return $o_{e} \mathbb{R}^{B\times L \times N}$ and $\hat{y}\in \mathbb{R}^{B\times L \times V}$ with : 

* $o_e = RMSNORM \left( b_h\circ \dots   \circ \circ b_2 \circ b_1 (w_{word\_embedding}x^\intercal) \right)$ with $b_i$ the $i^{th}$ block
* $\hat{y} = W_{lm\_head} o_e^{T}$



In [268]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocabulary_size, embed_size, intermediate_size, num_heads, hidden_layers=5, **kwargs):
        super().__init__(**kwargs)
        self.nh = num_heads
        self.nhl = hidden_layers
        self.es = embed_size 

        self.wte = nn.Embedding(vocabulary_size, embed_size)

        self.blocks = nn.ModuleList([TransformerDecoderBlock(embed_size, intermediate_size, num_heads) for _ in range(hidden_layers)])
        self.ln_f = nn.LayerNorm(embed_size, eps=1e-3)
        self.lm_head = nn.Linear(embed_size, vocabulary_size)


        
    def forward(
        self,
        input_ids,
        attention_mask=None,
        k_cache=None,
        v_cache=None,
        start_cache=0,
    ):
        input_embed = self.wte(input_ids)
        intermediate = input_embed

        # Cache containers for the first call
        if k_cache is None:
            k_cache = [None] * self.nhl
        if v_cache is None:
            v_cache = [None] * self.nhl

        new_k_cache = []
        new_v_cache = []

        for i, block in enumerate(self.blocks):
            intermediate, layer_k, layer_v = block(
                intermediate,
                attention_mask=attention_mask,
                k_cache=k_cache,
                v_cache=v_cache,
                start_cache=start_cache,
                layer=i,
            )

            new_k_cache.append(layer_k)
            new_v_cache.append(layer_v)

        output_embed = self.ln_f(intermediate)
        output_lm = self.lm_head(output_embed)

        return output_embed, output_lm, new_k_cache, new_v_cache

##  Training the transformer

### I.Dataset 
For simplicity we will use a small dataset name TinyStories, we can load it using the huggingface Datasets library as following

In [269]:
tinystories_dataset = load_dataset("roneneldan/TinyStories")
training_set = tinystories_dataset['train']
print(training_set[0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


### The Dataloader

We can define the batch_size default we will use, using the DataLoader object. Here we consider that the training machine will have enough RAM for a batch of 16 samples.

In [270]:
training_dl = DataLoader(training_set, batch_size=16, shuffle=True) 

### III. Tokenizer

In this exercice we will consider the LLama Tokenizer, however, you can train your own tokenizer if you prefer (see lab 2)

In [271]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model_hg_id = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_hg_id)
tokenizer.pad_token = '<pad>'
vocabulary_size = tokenizer.vocab_size

### IV. Create the model and the optimizer

In [272]:
device = 'cpu'
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 2)
model = model.train()
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=6e-4)


In [273]:
model

TransformerDecoder(
  (wte): Embedding(32768, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerDecoderBlock(
      (attention_module): RoPEAttentionModule(
        (Wq): Linear(in_features=128, out_features=512, bias=True)
        (Wk): Linear(in_features=128, out_features=512, bias=True)
        (Wv): Linear(in_features=128, out_features=512, bias=True)
        (Wo): Linear(in_features=512, out_features=128, bias=True)
      )
      (feed_forward): TransformerFeedForward(
        (gate_proj): Linear(in_features=128, out_features=256, bias=True)
        (up_proj): Linear(in_features=256, out_features=256, bias=True)
        (down_proj): Linear(in_features=256, out_features=128, bias=True)
        (gate_func): SiLU()
      )
      (attention_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
      (feed_forward_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
    )
  )
  (ln_f): LayerNorm((128,), eps=0.001, elementwise_affine=True)
  (lm_head): Linear(i

### VI. Training the model

In [ ]:
model = model.train()
avg_loss = []
start_time = time.time()

for i, data in enumerate(training_dl):
    res = tokenizer(data['text'], return_tensors="pt", padding=True, padding_side='right', max_length=128, truncation=True)
    x = res.input_ids[:, :-1] + 0
    y = res.input_ids[:, 1:] + 0
    # attention_mask = (res.attention_mask == 0)
    optimizer.zero_grad()
    oe, oy, _, _ = model(x.to(device))
    loss = loss_function(oy.to(device).view(-1, vocabulary_size), y.to(device).view(-1))
    loss.backward()
    loss_value = loss.item()
    avg_loss.append(loss_value)
    if( i%500 == 0):
        elapsed_time = time.time() - start_time
        remaining_time = int((elapsed_time/(i+1)) * (len(training_dl) - i))
        loop_sec = ((i+1)/elapsed_time)
        print(f"The loss at iteration {i+1} is {sum(avg_loss)/len(avg_loss):3.4f} remaining_time is {strftime('%H:%M:%S', gmtime(remaining_time))}s ({loop_sec:4.1f} it/s)", flush=True)
        avg_loss = []
    if( i%1000 == 0):
        torch.save(optimizer.state_dict() ,"optimizer_state_dict_llama_mini.pth.temp")
        torch.save(model.state_dict() ,"transformer_state_dict_llama_mini.pth.temp")
        shutil.copyfile("optimizer_state_dict_llama_mini.pth.temp", "optimizer_state_dict_llama_mini.pth")
        shutil.copyfile("transformer_state_dict_llama_mini.pth.temp", "transformer_state_dict_llama_mini.pth")
    optimizer.step()

In [274]:
state_dict = torch.load(
    "model/transformer_state_dict_llama_mini.pth",
    map_location=device,
    weights_only=True
)

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

TransformerDecoder(
  (wte): Embedding(32768, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerDecoderBlock(
      (attention_module): RoPEAttentionModule(
        (Wq): Linear(in_features=128, out_features=512, bias=True)
        (Wk): Linear(in_features=128, out_features=512, bias=True)
        (Wv): Linear(in_features=128, out_features=512, bias=True)
        (Wo): Linear(in_features=512, out_features=128, bias=True)
      )
      (feed_forward): TransformerFeedForward(
        (gate_proj): Linear(in_features=128, out_features=256, bias=True)
        (up_proj): Linear(in_features=256, out_features=256, bias=True)
        (down_proj): Linear(in_features=256, out_features=128, bias=True)
        (gate_func): SiLU()
      )
      (attention_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
      (feed_forward_layer_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
    )
  )
  (ln_f): LayerNorm((128,), eps=0.001, elementwise_affine=True)
  (lm_head): Linear(i

## Decoding time

There is different way to decode, the simplest one is the greedy decoding method, the principle is to loop with the new token produced each time (complete `nonefficient_greedy_decoding`) and choosing the token the most likely. We can also store Keys and Values as shown in course that in practice only accellerate the generation but not change it (complete `greedy_decoding`). And finally we can also sampling new tokens (complete `sampling_decoding` ).

In [293]:
def nonefficient_greedy_decoding(model, x, max_new_tokens=64):
    previous_gen = x
    for _ in range(max_new_tokens):
        with torch.no_grad():
            _, logits, _, _ = model(previous_gen)
            next_token_logits = logits[:, -1, :]
            next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)
            previous_gen = torch.cat(
                [previous_gen, next_token_id],
                dim=1
            )
    return previous_gen


def greedy_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=tokenizer):
    model.eval()
    previous_gen = x
    k_cache, v_cache = None, None
    start_cache = 0
    
    # Treating the entire prompt
    with torch.no_grad():
                start_cache = previous_gen.size(1) - 1
                _, logits, k_cache, v_cache = model(previous_gen, k_cache=k_cache, v_cache=v_cache, start_cache=start_cache)
    
    # Treating the next token using the cache
    for _ in range(max_new_tokens):
        next_token_logits = logits[:, -1, :]
        next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)
        previous_gen = torch.cat(
            [previous_gen, next_token_id],
            dim=1
        )
        start_cache = previous_gen.size(1) - 1

        _, logits, k_cache, v_cache = model(
            next_token_id,
            k_cache=k_cache,
            v_cache=v_cache,
            start_cache=start_cache,
        )
        
        # Limiting the used cache
        if max_cache_size is not None:
                k_cache = [
                    k[:, :, -max_cache_size:, :]
                    for k in k_cache
                ]
                v_cache = [
                    v[:, :, -max_cache_size:, :]
                    for v in v_cache
                ]
    
    return previous_gen
    

def sampling_decoding(model, x, max_new_tokens=64, max_cache_size=512, tokenizer=None, temperature=.7):
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    model.eval()

    previous_gen = x
    k_cache, v_cache = None, None
    start_cache = 0
    
    # Treating the entire prompt
    with torch.no_grad():
        start_cache = previous_gen.size(1) - 1
        _, logits, k_cache, v_cache = model(previous_gen, k_cache=k_cache, v_cache=v_cache, start_cache=start_cache)
    
    # Treating the next token using the cache
    for _ in range(max_new_tokens):
        next_token_logits = logits[:, -1, :]
        # We compute the scaled logit using the temprerature
        scaled_logits = next_token_logits / temperature
        # Computing the probablities
        probs = torch.softmax(scaled_logits, dim=-1)
        next_token_id = torch.multinomial(probs, num_samples=1)
        previous_gen = torch.cat(
            [previous_gen, next_token_id],
            dim=1
        )
        start_cache = previous_gen.size(1) - 1

        _, logits, k_cache, v_cache = model(
            next_token_id,
            k_cache=k_cache,
            v_cache=v_cache,
            start_cache=start_cache,
        )
        
        # Limiting the used cache
        if max_cache_size is not None:
                k_cache = [
                    k[:, :, -max_cache_size:, :]
                    for k in k_cache
                ]
                v_cache = [
                    v[:, :, -max_cache_size:, :]
                    for v in v_cache
                ]
    
    return previous_gen



In [313]:
# None-efficiemt greedy decoding
text = "Alice"

tokenized_text = tokenizer(text, return_tensors="pt")
input_ids = tokenized_text["input_ids"]

output_ids = nonefficient_greedy_decoding(
    model, input_ids, max_new_tokens=64
)
print(f"None-efficient greedy decoding: {tokenizer.decode(output_ids[0], skip_special_tokens=True)}")


None-efficient greedy decoding: Alice was a little boy who loved to play outside. One day, she saw a big, red ball in the park. She wanted to play with it, but she was too small.

"Can I play with the ball?" she asked.

"No, it's mine!" said Alice.



In [314]:
model = TransformerDecoder(vocabulary_size, 128, 256, 4, 2)
model.load_state_dict(torch.load("model/transformer_state_dict_llama_mini.pth", map_location="cpu"))
model = model.eval()

decoding_method = greedy_decoding

text = "Alice"
model.to("cpu")
tokenized_text = tokenizer(text, return_tensors='pt')

output_ids = decoding_method(model, tokenized_text['input_ids'], max_new_tokens=64, max_cache_size=512, tokenizer=tokenizer)[0]
print(f"Greedy decoding: \n{tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]}")

Greedy decoding: 
Alice was a little girl who liked to play outside. She had a big yard with lots of trees and flowers. She liked to swing on the swing and swing on the swing. She also liked to swing on the swing and swing.

One day, she saw a big dog with a long tail. The dog was


In [315]:
decoding_method = sampling_decoding

text = "Alice"
model.to("cpu")
tokenized_text = tokenizer(text, return_tensors='pt')

output_ids = decoding_method(model, tokenized_text.input_ids.to("cpu"), max_new_tokens=64)[0]
print(f"Sample decoding: {tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]}")

Sample decoding: Alice was happy. She had a big board in her room and she loved to play with it. She carried it everywhere she went. One day, a small boy came to her room and started to play with the board. He was so happy, she quickly set the board off. She had so much fun with her board


## Evaluation ?

It remains difficult to evaluate the model. You can here propose an evalaution based on perplexity using an other model, or comparing results to ground truth using the test (here only validation) set, but it will not be totally informative !!!

In [302]:
validation_dataset = tinystories_dataset['validation']

In [301]:
def evaluate_perplexity(model, input_ids):
    model.eval()
    device = next(model.parameters()).device
    input_ids = input_ids.to(device)

    with torch.no_grad():
        _, logits = model(input_ids)

        # Predict each next token
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )

        perplexity = torch.exp(loss)

    return loss.item(), perplexity.item()

In [307]:
from tqdm import tqdm
test_dl = DataLoader(
    validation_dataset,
    batch_size=16,
    shuffle=False,
)

model.eval()
total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in tqdm(test_dl):
        texts = batch["text"]

        input_ids = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        ).input_ids.to(device)

        _, logits, _, _ = model(input_ids)

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        # Exclude padding tokens from the loss
        pad_id = tokenizer.pad_token_id
        valid = shift_labels != pad_id if pad_id is not None else torch.ones_like(
            shift_labels, dtype=torch.bool
        )

        token_losses = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            reduction="none",
        ).reshape_as(shift_labels)

        total_loss += token_losses[valid].sum().item()
        total_tokens += valid.sum().item()

loss = total_loss / total_tokens
ppl = torch.exp(torch.tensor(loss)).item()

print(f"Test loss: {loss:.4f}")
print(f"Perplexity: {ppl:.2f}")

100%|██████████| 1375/1375 [03:56<00:00,  5.81it/s]

Test loss: 1.9003
Perplexity: 6.69


## Conclusion

In this lab, we implemented from scratch and evaluated a decoder-only Transformer for autoregressive text generation, including KV caching to improve inference efficiency. It took roughly 2 hours to train the model on a T4 GPU in Google Colab. rThe training loss goes from **10.55 to 1.89** on the last epoch (132001th), while the validation loss is **1.90** and a perplexity of **6.69** suggesting good generalisation to unseen data, although comparison with training perplexity and further evaluation would be needed to confirm it.

We compared three decoding approaches. **Greedy decoding** consistently selects the most probable next token, producing deterministic but sometimes repetitive stories. The second decoding approach also generated coherent text but showed limitations in consistency. **Sampling-based decoding** introduced randomness, resulting in more varied and creative stories, although sometimes at the cost of coherence. Finally, KV caching improved generation efficiency by reusing previously computed keys and values instead of recomputing them at every step.

Overall, this lab demonstrated the trade-off between generation quality, diversity, and computational efficiency, while connecting Transformer architecture, language-model evaluation, and practical text generation.
